# PAD-ONAP — S3 AI Pipeline: Full Training on CICDDoS2019

**Thesis:** Proactive AI-Driven DDoS Defense (PAD-ONAP)  
**Module:** M2 — AI Detection & Forecast  
**Tracks:**
- **Track A:** XGBoost 12-class classifier (22 CICFlowMeter features, Extra Trees selection)
- **Track B:** LSTM 3-horizon forecaster (+1m/+5m/+15m, 60-minute lookback)

**Dataset:** CICDDoS2019 (Canadian Institute for Cybersecurity)  
**GPU:** RTX T4 / P100 on Kaggle

---

## Pipeline Flow
```
CICDDoS2019 CSVs
     │
     ▼  [Section 1] Dataset Audit
Row counts, label distribution, class mapping
     │
     ▼  [Section 2] Feature Extraction (streaming, full dataset)
22 CICFlowMeter features per 5s window → X_train, X_test, y_train, y_test
     │
     ├──▶ [Section 3] Track A: XGBoost 12-class → xgboost_v3.json
     │
     └──▶ [Section 4] Track B: LSTM 3-horizon → lstm_v3.pt
     │
     ▼  [Section 5] Evaluation Summary + SHAP
```

---

## 🔧 Notes (pipeline-aligned v4)

- **FIX-1b** `Cell 5`: Adaptive BENIGN step — dùng `BENIGN_STEP=5` khi gặp window BENIGN thay vì `STEP=50`, tăng quota BENIGN lên `MIN_BENIGN_WINDOWS=30_000`.
- **FIX-1c** `Cell 6`: Per-segment stratified temporal split — cắt 80/20 bên trong từng file-segment để train/test có cùng phân phối class (tránh SYN test > SYN train).
- **ALIGN-B** Track B dùng LSTM 3-horizon (t+1/t+5/t+15) theo Pipeline; bỏ multi-class forecast 4-horizon.

In [1]:
# ── Cell 0: Install / upgrade packages ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('xgboost>=2.0', 'shap>=0.44', 'scikit-learn>=1.4', 'optuna>=3.6', 'imbalanced-learn>=0.12')
print('Packages ready.')


Packages ready.


In [2]:
# ── Cell 1: Imports & GPU check ──────────────────────────────────────────────
import os, glob, warnings, time, json, pickle
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import xgboost as xgb
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    balanced_accuracy_score, average_precision_score,
    )
warnings.filterwarnings('ignore')

# ── Paths (adjust DATASET_DIR to your Kaggle dataset mount point) ────────────
DATASET_DIR  = '/kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files'
OUTPUT_DIR   = Path('/kaggle/working/pad_onap_v3')
MODELS_DIR   = OUTPUT_DIR / 'models'
DATA_DIR     = OUTPUT_DIR / 'processed'
FIGURES_DIR  = OUTPUT_DIR / 'figures'

for d in [MODELS_DIR, DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── GPU ───────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

XGB_DEVICE = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
try:
    _m = xgb.XGBClassifier(device='cuda', n_estimators=3, verbosity=0)
    _m.fit(np.random.randn(50,5), np.random.randint(0,3,50))
    print('XGBoost GPU: OK')
except Exception:
    XGB_DEVICE = 'cpu'
    print('XGBoost GPU: not available, using CPU')

PyTorch device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
XGBoost GPU: OK


In [3]:
# ── Cell 1.5: Run mode switch (tune vs baseline) ─────────────────────────────
RUN_MODE = 'tune'  # 'tune' | 'baseline'

if RUN_MODE not in {'tune', 'baseline'}:
    raise ValueError("RUN_MODE must be either 'tune' or 'baseline'")

if RUN_MODE == 'tune':
    # Run full search once and save best configs.
    RUN_XGB_TUNING = True
    LOAD_XGB_TUNED = False
    SAVE_XGB_TUNED = True
else:
    # Fast baseline: load best tuned configs and skip search.
    RUN_XGB_TUNING = False
    LOAD_XGB_TUNED = True
    SAVE_XGB_TUNED = False

# Track B LSTM tuning hooks (disabled by default)
RUN_LSTM_TUNING = True
LOAD_LSTM_TUNED = False
SAVE_LSTM_TUNED = True

print(f'RUN_MODE={RUN_MODE}')
print(f'  XGB -> tune:{RUN_XGB_TUNING}, load:{LOAD_XGB_TUNED}, save:{SAVE_XGB_TUNED}')
print(f'  LSTM -> tune:{RUN_LSTM_TUNING}, load:{LOAD_LSTM_TUNED}, save:{SAVE_LSTM_TUNED}')

RUN_MODE=tune
  XGB -> tune:True, load:False, save:True
  LSTM -> tune:True, load:False, save:True


---
## Section 1 — Dataset Audit

In [4]:
# ── Cell 2: Dataset Audit (full scan) ────────────────────────────────────────
LABEL_MAP = {
    'BENIGN': 0,
    'DrDoS_DNS': 1,
    'DrDoS_LDAP': 2, 'LDAP': 2,
    'DrDoS_MSSQL': 3, 'MSSQL': 3,
    'DrDoS_NetBIOS': 4, 'NetBIOS': 4,
    'DrDoS_NTP': 5, 'NTP': 5,
    'DrDoS_SNMP': 6, 'SNMP': 6,
    'DrDoS_SSDP': 7, 'SSDP': 7,
    'DrDoS_UDP': 8, 'UDP': 8,
    'Syn': 9, 'SYN': 9,
    'UDP-lag': 10, 'UDP-Lag': 10, 'UDPLag': 10,
    'WebDDoS': 11, 'Web DDoS': 11,
}

CLASS_NAMES = {
    0: 'BENIGN',
    1: 'DrDoS_DNS',
    2: 'DrDoS_LDAP',
    3: 'DrDoS_MSSQL',
    4: 'DrDoS_NetBIOS',
    5: 'DrDoS_NTP',
    6: 'DrDoS_SNMP',
    7: 'DrDoS_SSDP',
    8: 'DrDoS_UDP',
    9: 'Syn',
    10: 'UDP-lag',
    11: 'WebDDoS',
}
N_CLASSES = len(CLASS_NAMES)

FEATURE_NAMES = [
    'flow_duration',
    'total_fwd_packets',
    'total_bwd_packets',
    'total_length_fwd_packets',
    'total_length_bwd_packets',
    'fwd_packet_length_max',
    'fwd_packet_length_mean',
    'bwd_packet_length_mean',
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'flow_iat_mean',
    'flow_iat_std',
    'fwd_iat_total',
    'fwd_iat_mean',
    'bwd_iat_total',
    'syn_flag_count',
    'ack_flag_count',
    'fwd_psh_flags',
    'init_win_bytes_fwd',
    'init_win_bytes_bwd',
    'min_seg_size_fwd',
    'protocol',
]

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files')
print(f'Total size: {sum(os.path.getsize(f) for f in csv_files)/1e9:.2f} GB\n')

all_labels   = defaultdict(int)
file_summary = []
CHUNKSIZE    = 500_000

for f in csv_files:
    fname     = os.path.basename(f)
    file_rows = 0
    file_lbls = defaultdict(int)
    try:
        for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines='skip'):
            chunk.columns = [c.strip() for c in chunk.columns]
            lcol = next((c for c in chunk.columns if c.lower() == 'label'), None)
            if lcol is None: continue
            for lbl, cnt in chunk[lcol].str.strip().value_counts().items():
                file_lbls[lbl] += int(cnt)
                all_labels[lbl] += int(cnt)
            file_rows += len(chunk)
        benign_cnt = file_lbls.get('BENIGN', 0)
        attack_cnt = file_rows - benign_cnt
        lbl_str = ', '.join(f'{k}:{v//1000}k' for k,v in file_lbls.items())
        print(f'  {fname:<35} rows={file_rows:>10,}  BENIGN={benign_cnt:>8,}  attack={attack_cnt:>10,}')
        file_summary.append({'file': fname, 'rows': file_rows, 'benign': benign_cnt, 'attack': attack_cnt})
    except Exception as e:
        print(f'  ERROR {fname}: {e}')

grand_total  = sum(x['rows']    for x in file_summary)
grand_benign = sum(x['benign']  for x in file_summary)
grand_attack = sum(x['attack']  for x in file_summary)
print(f'\nTOTAL: {grand_total:,} rows | BENIGN={grand_benign:,} | Attack={grand_attack:,}')

Found 18 CSV files
Total size: 31.06 GB

  DrDoS_DNS.csv                       rows= 5,074,413  BENIGN=   3,402  attack= 5,071,011
  DrDoS_LDAP.csv                      rows= 2,181,542  BENIGN=   1,612  attack= 2,179,930
  DrDoS_MSSQL.csv                     rows= 4,524,498  BENIGN=   2,006  attack= 4,522,492
  DrDoS_NTP.csv                       rows= 1,217,007  BENIGN=  14,365  attack= 1,202,642
  DrDoS_NetBIOS.csv                   rows= 4,094,986  BENIGN=   1,707  attack= 4,093,279
  DrDoS_SNMP.csv                      rows= 5,161,377  BENIGN=   1,507  attack= 5,159,870
  DrDoS_SSDP.csv                      rows= 2,611,374  BENIGN=     763  attack= 2,610,611
  DrDoS_UDP.csv                       rows= 3,136,802  BENIGN=   2,157  attack= 3,134,645
  Syn.csv                             rows= 1,582,681  BENIGN=     392  attack= 1,582,289
  TFTP.csv                            rows=20,107,827  BENIGN=  25,247  attack=20,082,580
  UDPLag.csv                          rows=   370,605  BENI

In [5]:
# ── Cell 3: Class distribution table ──────────────────────────────────────────
class_counts = defaultdict(int)
for lbl, cnt in all_labels.items():
    cls = LABEL_MAP.get(lbl, -1)
    if cls >= 0:
        class_counts[cls] += cnt

print('=== CLASS DISTRIBUTION (spec mapping) ===')
print(f'{"Class":<5} {"Name":<16} {"Rows":>14} {"Pct":>7}  {"Est. Windows (step=50)":>22}')
print('-' * 75)
for cls_id in range(N_CLASSES):
    cnt  = class_counts.get(cls_id, 0)
    pct  = 100 * cnt / max(grand_total, 1)
    wins = max(0, (cnt - 100) // 50) if cnt >= 100 else 0
    print(f'[{cls_id}]   {CLASS_NAMES[cls_id]:<16} {cnt:>14,} {pct:>6.2f}%  {wins:>22,}')
print('-' * 75)
print(f"{''  :5} {'TOTAL':<16} {grand_total:>14,} {100:>6.2f}%")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [CLASS_NAMES[i] for i in range(N_CLASSES)]
counts = [class_counts.get(i, 0) for i in range(N_CLASSES)]
colors = ['#2196F3','#F44336','#FF9800','#4CAF50','#9C27B0','#00BCD4','#795548','#3F51B5','#009688','#FFC107','#8BC34A','#E91E63']

axes[0].barh(names, counts, color=colors[:len(names)])
axes[0].set_title('Row Count per Class (raw)')
axes[0].set_xlabel('Number of rows')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))

non_zero = [(n, c) for n, c in zip(names, counts) if c > 0]
axes[1].pie([c for _,c in non_zero], labels=[n for n,_ in non_zero],
            colors=colors[:len(non_zero)], autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Distribution (row %)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_class_distribution_raw.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/01_class_distribution_raw.png')

=== CLASS DISTRIBUTION (spec mapping) ===
Class Name                       Rows     Pct  Est. Windows (step=50)
---------------------------------------------------------------------------
[0]   BENIGN                  113,828   0.16%                   2,274
[1]   DrDoS_DNS             5,071,011   7.20%                 101,418
[2]   DrDoS_LDAP            4,095,052   5.81%                  81,899
[3]   DrDoS_MSSQL          10,309,945  14.64%                 206,196
[4]   DrDoS_NetBIOS         7,750,776  11.01%                 155,013
[5]   DrDoS_NTP             1,202,642   1.71%                  24,050
[6]   DrDoS_SNMP            5,159,870   7.33%                 103,195
[7]   DrDoS_SSDP            2,610,611   3.71%                  52,210
[8]   DrDoS_UDP             7,001,800   9.94%                 140,034
[9]   Syn                   6,473,789   9.19%                 129,473
[10]   UDP-lag                 368,334   0.52%                   7,364
[11]   WebDDoS                     439   

---
## Section 2 — Feature Extraction (Streaming, Full Dataset)

In [6]:
# ── Cell 4: Feature extraction helpers ───────────────────────────────────────
def extract_22_features(df_window: pd.DataFrame) -> np.ndarray:
    """Compute 22 CICFlowMeter features (window-level mean) for Track A."""
    n = len(df_window)
    if n == 0:
        return np.zeros(22, dtype=np.float32)

    def col(name):
        if name in df_window.columns:
            vals = pd.to_numeric(df_window[name], errors='coerce').values
            vals = np.where(np.isinf(vals) | (vals < 0), np.nan, vals)
            if np.isnan(vals).all():
                return np.zeros(n)
            median_val = np.nanmedian(vals)
            return np.where(np.isnan(vals), median_val, vals)
        return np.zeros(n)

    def mean(name):
        return float(np.nanmean(col(name)))

    proto_vals = col('Protocol').astype(int)
    if len(proto_vals) == 0:
        proto_mode = 0
    else:
        proto_mode = int(np.bincount(proto_vals).argmax())

    feat = np.array([
        mean('Flow Duration'),
        mean('Total Fwd Packets'),
        mean('Total Backward Packets'),
        mean('Total Length of Fwd Packets'),
        mean('Total Length of Bwd Packets'),
        mean('Fwd Packet Length Max'),
        mean('Fwd Packet Length Mean'),
        mean('Bwd Packet Length Mean'),
        mean('Flow Bytes/s'),
        mean('Flow Packets/s'),
        mean('Flow IAT Mean'),
        mean('Flow IAT Std'),
        mean('Fwd IAT Total'),
        mean('Fwd IAT Mean'),
        mean('Bwd IAT Total'),
        mean('SYN Flag Count'),
        mean('ACK Flag Count'),
        mean('Fwd PSH Flags'),
        mean('Init_Win_bytes_forward'),
        mean('Init_Win_bytes_backward'),
        mean('min_seg_size_forward'),
        float(proto_mode),
    ], dtype=np.float32)

    return np.nan_to_num(feat, nan=0.0, posinf=1e6, neginf=0.0)

print('Feature helpers ready.')

Feature helpers ready.


In [8]:
# ── Cell 5: Streaming mixed-temporal feature extraction — v4 (FIX-1b) ────────
# FIX-1b: Adaptive BENIGN step — khi gặp đoạn BENIGN trong file, dùng STEP nhỏ
#          hơn (BENIGN_STEP=5) thay vì STEP=50 để tạo nhiều window BENIGN hơn.
#         MIN_BENIGN_WINDOWS nâng từ 20_000 → 30_000 để đảm bảo quota thực tế.

WINDOW_SIZE        = 100
STEP               = 50       # bước mặc định cho attack
BENIGN_STEP        = 5        # bước nhỏ hơn khi majority row là BENIGN
MAX_WINDOWS        = 240_000
MIN_CLASS_WINDOWS  = 20_000
MIN_BENIGN_WINDOWS = 20_000   # FIX-1b: quota riêng cho BENIGN

mixed_windows_X        = []
mixed_windows_y        = []
mixed_segment_lengths  = []
total_rows_read        = 0
unknown_rows           = 0
t_start                = time.time()


def window_majority_label(labels: np.ndarray) -> int:
    counts     = np.bincount(labels, minlength=N_CLASSES)
    top        = counts.max()
    candidates = np.where(counts == top)[0]
    if len(candidates) == 1:
        return int(candidates[0])
    return int(labels[-1])


def build_class_quota(max_windows: int, min_per_class: int, min_benign: int):
    """Tính quota với ưu tiên riêng cho BENIGN (class 0)."""
    class_rows = {i: int(class_counts.get(i, 0)) for i in range(N_CLASSES)}
    active     = [c for c, n in class_rows.items() if n > 0]
    if not active:
        active = sorted({int(v) for v in LABEL_MAP.values() if int(v) >= 0})

    quota = {c: 0 for c in range(N_CLASSES)}
    k     = len(active)
    if k == 0:
        return quota, []

    # BENIGN riêng nếu nằm trong active
    if 0 in active:
        quota[0]   = min_benign
        non_benign = [c for c in active if c != 0]
        remain     = max_windows - min_benign
    else:
        non_benign = active
        remain     = max_windows

    # Phần còn lại chia cho các class khác
    k2       = len(non_benign)
    if k2 > 0:
        base = min_per_class
        min_total = base * k2
        if min_total >= remain:
            base2 = remain // k2
            rem2  = remain - base2 * k2
            for i, c in enumerate(non_benign):
                quota[c] = base2 + (1 if i < rem2 else 0)
        else:
            for c in non_benign:
                quota[c] = base
            leftover = remain - min_total
            weights  = np.array([max(class_rows[c], 1) for c in non_benign], dtype=np.float64)
            weights  = weights / weights.sum()
            raw      = leftover * weights
            floor_a  = np.floor(raw).astype(int)
            frac     = raw - floor_a
            for i, c in enumerate(non_benign):
                quota[c] += int(floor_a[i])
            rem3  = int(leftover - floor_a.sum())
            order = np.argsort(-frac)
            for i in range(rem3):
                quota[non_benign[order[i % k2]]] += 1

    return quota, active


class_quota, active_classes = build_class_quota(MAX_WINDOWS, MIN_CLASS_WINDOWS, MIN_BENIGN_WINDOWS)
class_windows = defaultdict(int)

print(f'Extracting mixed temporal windows from {len(csv_files)} CSV files...')
print(f'WINDOW_SIZE={WINDOW_SIZE}, STEP={STEP}, BENIGN_STEP={BENIGN_STEP}, MAX_WINDOWS={MAX_WINDOWS}')
print(f'Active classes: {[CLASS_NAMES.get(c, c) for c in active_classes]}')
print('Class quota plan:')
for c in active_classes:
    print(f'  - {CLASS_NAMES.get(c, c):<15}: {class_quota[c]:>7,} windows')
print()

for fi, f in enumerate(csv_files):
    if len(mixed_windows_X) >= MAX_WINDOWS:
        break

    remaining_slots = MAX_WINDOWS - len(mixed_windows_X)
    remaining_files = max(1, len(csv_files) - fi)
    file_quota      = int(np.ceil(remaining_slots / remaining_files))

    fname                  = os.path.basename(f)
    file_rows              = 0
    file_windows           = 0
    file_windows_by_class  = defaultdict(int)
    carry_over             = pd.DataFrame()

    try:
        for chunk in pd.read_csv(f, chunksize=200_000, low_memory=False, on_bad_lines='skip'):
            chunk.columns = [c.strip() for c in chunk.columns]
            lcol = next((c for c in chunk.columns if c.lower() == 'label'), None)
            if lcol is None:
                continue

            mapped       = chunk[lcol].str.strip().map(LABEL_MAP)
            unknown_rows += int(mapped.isna().sum())
            chunk['_class'] = mapped
            chunk = chunk.dropna(subset=['_class'])
            if len(chunk) == 0:
                continue
            chunk['_class'] = chunk['_class'].astype(int)

            file_rows       += len(chunk)
            total_rows_read += len(chunk)

            df_combined = (
                pd.concat([carry_over, chunk], ignore_index=True)
                if len(carry_over) > 0 else chunk.reset_index(drop=True)
            )

            i = 0
            while i + WINDOW_SIZE <= len(df_combined):
                if len(mixed_windows_X) >= MAX_WINDOWS or file_windows >= file_quota:
                    break

                window = df_combined.iloc[i:i + WINDOW_SIZE]
                lbl    = window_majority_label(window['_class'].values.astype(np.int32))

                # FIX-1b: dùng bước nhỏ khi window thuộc BENIGN để tạo nhiều mẫu hơn
                step_used = BENIGN_STEP if lbl == 0 else STEP

                remaining_quota = sum(max(0, class_quota[c] - class_windows[c]) for c in active_classes)
                remaining_need  = MAX_WINDOWS - len(mixed_windows_X)
                class_allowed   = (
                    class_windows[lbl] < class_quota.get(lbl, 0)
                    or remaining_quota < remaining_need
                )

                if class_allowed:
                    feats = extract_22_features(window)
                    mixed_windows_X.append(feats)
                    mixed_windows_y.append(lbl)
                    class_windows[lbl]          += 1
                    file_windows_by_class[lbl]  += 1
                    file_windows                += 1

                i += step_used

            carry_over = df_combined.iloc[i:].reset_index(drop=True)
            if len(mixed_windows_X) >= MAX_WINDOWS or file_windows >= file_quota:
                break

        if file_windows > 0:
            mixed_segment_lengths.append(file_windows)

        elapsed = time.time() - t_start
        if file_windows > 0:
            parts          = [f"{CLASS_NAMES.get(c,'Unknown')}: {cnt}"
                              for c, cnt in sorted(file_windows_by_class.items())]
            class_dist_str = ' | '.join(parts)
        else:
            class_dist_str = '0 windows'

        print(
            f'  [{fi+1:2d}/{len(csv_files)}] {fname:<35} rows={file_rows:>8,}\n',
            f'       └─ FileQuota={file_quota:>6,} | Windows={file_windows:>7,} | Total={len(mixed_windows_X):>8,} | Time={elapsed:.0f}s\n',
            f'       └─ Classes: {class_dist_str}\n',
        )
    except Exception as e:
        print(f'  ERROR {fname}: {e}')

if len(mixed_windows_X) == 0:
    raise ValueError('No mixed windows extracted. Please check LABEL_MAP and input CSV files.')

X_mixed               = np.stack(mixed_windows_X).astype(np.float32)
y_mixed               = np.array(mixed_windows_y, dtype=np.int32)
mixed_segment_lengths = [int(x) for x in mixed_segment_lengths if x > 0]

final_classes, final_counts = np.unique(y_mixed, return_counts=True)
final_dist = {CLASS_NAMES.get(c, c): int(cnt) for c, cnt in zip(final_classes, final_counts)}

print('=' * 60)
print('EXTRACTION SUMMARY:')
print(f'Total rows read          : {total_rows_read:,}')
print(f'Unknown/unmapped rows    : {unknown_rows:,}')
print(f'Extracted windows        : {len(X_mixed):,}')
print(f'FINAL Class dist: {final_dist}')
print('Class quota usage:')
for c in active_classes:
    used = int(class_windows[c]); q = int(class_quota[c])
    pct  = used / max(q, 1) * 100
    print(f'  - {CLASS_NAMES.get(c, c):<15}: {used:>7,} / {q:>7,}  ({pct:.1f}%)')
print('=' * 60)


Extracting mixed temporal windows from 18 CSV files...
WINDOW_SIZE=100, STEP=50, BENIGN_STEP=5, MAX_WINDOWS=240000
Active classes: ['BENIGN', 'DrDoS_DNS', 'DrDoS_LDAP', 'DrDoS_MSSQL', 'DrDoS_NetBIOS', 'DrDoS_NTP', 'DrDoS_SNMP', 'DrDoS_SSDP', 'DrDoS_UDP', 'Syn', 'UDP-lag', 'WebDDoS']
Class quota plan:
  - BENIGN         :  20,000 windows
  - DrDoS_DNS      :  20,000 windows
  - DrDoS_LDAP     :  20,000 windows
  - DrDoS_MSSQL    :  20,000 windows
  - DrDoS_NetBIOS  :  20,000 windows
  - DrDoS_NTP      :  20,000 windows
  - DrDoS_SNMP     :  20,000 windows
  - DrDoS_SSDP     :  20,000 windows
  - DrDoS_UDP      :  20,000 windows
  - Syn            :  20,000 windows
  - UDP-lag        :  20,000 windows
  - WebDDoS        :  20,000 windows

  [ 1/18] DrDoS_DNS.csv                       rows= 800,000
        └─ FileQuota=13,334 | Windows= 13,334 | Total=  13,334 | Time=65s
        └─ Classes: BENIGN: 170 | DrDoS_DNS: 13164

  [ 2/18] DrDoS_LDAP.csv                      rows= 800,000
       

In [11]:
# ── Cell 6: Per-segment stratified temporal split — v4 (FIX-1c) ──────────────
# FIX-1c: Thay vì cắt ngang toàn bộ dữ liệu (tạo test set lệch class), ta split
#          80/20 BÊN TRONG từng file-segment. Train và test do đó có cùng phân
#          phối class. Purge PURGE_WINDOWS cửa sổ ở ranh giới để tránh data leak.

TEMPORAL_SPLIT_RATIO  = 0.8
PURGE_WINDOWS         = 10
MAX_BENIGN_OVERSAMPLE = 20
rng_fix               = np.random.default_rng(42)

if 'X_mixed' not in globals() or 'y_mixed' not in globals():
    raise ValueError('X_mixed/y_mixed not found. Run Cell 5 first.')

# ── Bước 1: split từng segment riêng (FIX-1c) ─────────────────────────────────
tr_X_parts, tr_y_parts = [], []
te_X_parts, te_y_parts = [], []
tr_segs, te_segs = [], []

cursor = 0
for seg_len in mixed_segment_lengths:
    end   = cursor + int(seg_len)
    split = cursor + max(1, int(seg_len * TEMPORAL_SPLIT_RATIO))
    purge = min(PURGE_WINDOWS, max(0, (end - split) - 1))

    tr_end   = split
    te_start = split + purge

    if tr_end > cursor:
        tr_X_parts.append(X_mixed[cursor:tr_end])
        tr_y_parts.append(y_mixed[cursor:tr_end])
        tr_segs.append(int(tr_end - cursor))

    if te_start < end:
        te_X_parts.append(X_mixed[te_start:end])
        te_y_parts.append(y_mixed[te_start:end])
        te_segs.append(int(end - te_start))

    cursor = end

if not tr_X_parts:
    raise ValueError('Train split vide — kiểm tra mixed_segment_lengths')
if not te_X_parts:
    raise ValueError('Test split vide — kiểm tra mixed_segment_lengths')

X_train_raw_base = np.vstack(tr_X_parts).astype(np.float32)
y_train_base     = np.hstack(tr_y_parts).astype(np.int32)
X_test_raw_base  = np.vstack(te_X_parts).astype(np.float32)
y_test_base      = np.hstack(te_y_parts).astype(np.int32)

# ── Bước 2: BENIGN oversampling trên train ─────────────────────────────────────
total_benign_rows = int(grand_benign) if 'grand_benign' in globals() else int((y_mixed == 0).sum())
raw_benign_X      = X_mixed[y_mixed == 0].astype(np.float32)
n_raw             = len(raw_benign_X)

bn_mask        = (y_train_base == 0)
atk_mask       = ~bn_mask
X_bn_tr_base   = X_train_raw_base[bn_mask]
X_atk_tr_base  = X_train_raw_base[atk_mask]
y_atk_tr_base  = y_train_base[atk_mask]

if len(X_bn_tr_base) == 0:
    raise ValueError('Không có mẫu BENIGN trong train — kiểm tra extraction.')
if len(X_atk_tr_base) == 0:
    raise ValueError('Không có mẫu attack trong train.')

target_bn_tr = min(len(X_atk_tr_base), len(X_bn_tr_base) * MAX_BENIGN_OVERSAMPLE)
bn_idx       = rng_fix.choice(len(X_bn_tr_base), size=target_bn_tr,
                               replace=(len(X_bn_tr_base) < target_bn_tr))
X_bn_tr = X_bn_tr_base[bn_idx]
y_bn_tr = np.zeros(len(X_bn_tr), dtype=np.int32)

X_train_raw = np.vstack([X_bn_tr, X_atk_tr_base]).astype(np.float32)
y_train     = np.hstack([y_bn_tr, y_atk_tr_base]).astype(np.int32)

# Test: phân phối tự nhiên, không oversample
X_test_raw = X_test_raw_base.astype(np.float32)
y_test     = y_test_base.astype(np.int32)

# ── Bước 3: shuffle nội bộ (không xáo trộn giữa train↔test) ──────────────────
s1           = rng_fix.permutation(len(X_train_raw))
X_train_raw, y_train = X_train_raw[s1], y_train[s1]
s2           = rng_fix.permutation(len(X_test_raw))
X_test_raw, y_test = X_test_raw[s2], y_test[s2]

# ── Bước 4: Scale ──────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

# ── In kết quả ────────────────────────────────────────────────────────────────
print('=== FIX-1c: Per-segment stratified temporal split ===')
print(f'Segments train: {len(tr_segs)} | test: {len(te_segs)}')
print(f'BENIGN train (gốc): {len(X_bn_tr_base):,} → sau oversample: {len(X_bn_tr):,} (x{MAX_BENIGN_OVERSAMPLE})')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

train_dist = {CLASS_NAMES.get(i, i): int((y_train == i).sum()) for i in np.unique(y_train)}
test_dist  = {CLASS_NAMES.get(i, i): int((y_test  == i).sum()) for i in np.unique(y_test)}
print(f'Train class dist : {train_dist}')
print(f'Test  class dist : {test_dist}')

# So sánh tỉ lệ để kiểm tra cân bằng
print('\nTỉ lệ train vs test (không tính BENIGN oversample):')
for cls in np.unique(y_train_base):
    n_tr = int((y_train_base == cls).sum())
    n_te = int((y_test_base  == cls).sum())
    print(f'  {CLASS_NAMES.get(cls, cls):<15}: train={n_tr:>6,}  test={n_te:>6,}  '
          f'ratio={n_tr/(n_tr+n_te)*100:.1f}%/{n_te/(n_tr+n_te)*100:.1f}%')

# Lưu dữ liệu
np.save(DATA_DIR / 'X_train.npy', X_train)
np.save(DATA_DIR / 'X_test.npy',  X_test)
np.save(DATA_DIR / 'y_train.npy', y_train)
np.save(DATA_DIR / 'y_test.npy',  y_test)
np.save(DATA_DIR / 'X_benign_raw.npy', raw_benign_X)
with open(DATA_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'\nSaved to {DATA_DIR}')

metadata = {
    'fix_version'        : 'v4_fix1b1c',
    'total_rows_read'    : int(total_rows_read),
    'total_benign_rows'  : int(total_benign_rows),
    'raw_benign_windows' : int(n_raw),
    'window_size'        : int(WINDOW_SIZE),
    'step'               : int(STEP),
    'benign_step'        : int(BENIGN_STEP),
    'n_features'         : int(len(FEATURE_NAMES)),
    'feature_names'      : FEATURE_NAMES,
    'split_strategy'     : 'per_segment_stratified_temporal_v4',
    'temporal_split_ratio': float(TEMPORAL_SPLIT_RATIO),
    'purge_windows'      : int(PURGE_WINDOWS),
    'max_benign_oversample': int(MAX_BENIGN_OVERSAMPLE),
    'max_windows'        : int(MAX_WINDOWS),
    'train_samples'      : int(len(X_train)),
    'test_samples'       : int(len(X_test)),
    'class_dist_train'   : {str(k): v for k, v in train_dist.items()},
    'class_dist_test'    : {str(k): v for k, v in test_dist.items()},
}
with open(DATA_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved to {DATA_DIR}')


=== FIX-1c: Per-segment stratified temporal split ===
Segments train: 18 | test: 18
BENIGN train (gốc): 14,730 → sau oversample: 126,564 (x20)
X_train: (253128, 22) | X_test: (35151, 22)
Train class dist : {'BENIGN': 126564, 'DrDoS_DNS': 10497, 'DrDoS_LDAP': 15085, 'DrDoS_MSSQL': 15982, 'DrDoS_NetBIOS': 16799, 'DrDoS_NTP': 7888, 'DrDoS_SNMP': 10601, 'DrDoS_SSDP': 10666, 'DrDoS_UDP': 15934, 'Syn': 17333, 'UDP-lag': 5779}
Test  class dist : {'BENIGN': 4148, 'DrDoS_DNS': 2657, 'DrDoS_LDAP': 4895, 'DrDoS_MSSQL': 3998, 'DrDoS_NetBIOS': 3181, 'DrDoS_NTP': 2657, 'DrDoS_SNMP': 2657, 'DrDoS_SSDP': 2657, 'DrDoS_UDP': 4046, 'Syn': 2657, 'UDP-lag': 1598}

Tỉ lệ train vs test (không tính BENIGN oversample):
  BENIGN         : train=14,730  test= 4,148  ratio=78.0%/22.0%
  DrDoS_DNS      : train=10,497  test= 2,657  ratio=79.8%/20.2%
  DrDoS_LDAP     : train=15,085  test= 4,895  ratio=75.5%/24.5%
  DrDoS_MSSQL    : train=15,982  test= 3,998  ratio=80.0%/20.0%
  DrDoS_NetBIOS  : train=16,799  test= 3

In [12]:
# ── Cell 7: Post-extraction statistics ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_counts = {CLASS_NAMES[i]: int((y_train==i).sum()) for i in range(N_CLASSES) if (y_train==i).sum()>0}
test_counts  = {CLASS_NAMES[i]: int((y_test==i).sum())  for i in range(N_CLASSES) if (y_test==i).sum()>0}

x_tr = list(train_counts.keys())
axes[0].bar(x_tr, list(train_counts.values()), color='steelblue')
axes[0].set_title('Train Set — Class Distribution')
axes[0].set_ylabel('Windows'); axes[0].tick_params(axis='x', rotation=30)

x_te = list(test_counts.keys())
axes[1].bar(x_te, list(test_counts.values()), color='coral')
axes[1].set_title('Test Set — Class Distribution')
axes[1].set_ylabel('Windows'); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_train_test_dist.png', dpi=150)
plt.show()

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print(f'\nFeature stats (train):')
df_stats = pd.DataFrame(X_train, columns=FEATURE_NAMES).describe().T
print(X_train)
display(df_stats[['mean','std','min','max']].round(4))

Train: (253128, 22)
Test:  (35151, 22)

Feature stats (train):
[[-0.16372848 -0.11338385 -0.15892741 ... -0.4751236  -0.1309825
  -1.1138873 ]
 [ 0.45936245 -0.03645001  0.26696953 ...  0.9883493  -0.10832693
  -1.1138873 ]
 [ 0.0381843  -0.16324042 -0.19809993 ... -0.3695535  -0.16109782
  -1.1138873 ]
 ...
 [ 1.1685933   0.02716011  0.327812   ...  0.46360248 -0.15170406
   0.8977568 ]
 [ 0.4150994  -0.15249546  0.04693666 ...  1.3594769  -0.1351268
   0.8977568 ]
 [ 0.319963   -0.07470202  0.20195983 ... -0.11689006 -0.14700717
   0.8977568 ]]


,mean,std,min,max
flow_duration,-0.0,1.0004,-0.7345,6.2190
total_fwd_packets,-0.0,1.0002,-0.2930,42.6958
total_bwd_packets,-0.0,0.9996,-0.3781,27.2610
total_length_fwd_packets,-0.0,0.9999,-0.3717,14.4493
total_length_bwd_packets,-0.0,0.9998,-0.1784,28.4253
fwd_packet_length_max,-0.0,1.0000,-0.8912,2.5348
fwd_packet_length_mean,0.0,1.0002,-0.7150,2.5065
bwd_packet_length_mean,-0.0,1.0005,-0.7142,7.6262
flow_bytes_per_sec,-0.0,1.0001,-0.5734,3.4219
flow_packets_per_sec,0.0,1.0000,-1.1472,2.0120


---
## Section 3 — Track A: XGBoost 12-class Classifier

In [16]:
# ── Cell 8: Hybrid augmentation — SMOTE for minority + Gaussian for rest ──────
# Goal: lift F1 of weak classes (DrDoS_SSDP, UDP-lag, DrDoS_NetBIOS, DrDoS_UDP)
# Strategy:
#   1) SMOTE oversample minority classes up to a target floor (synthesize realistic
#      samples in feature space — far better than copy-paste for ~1.5k-sample classes).
#   2) Add Gaussian noise on attack samples for regularisation.
from collections import Counter

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except Exception as _e:
    print(f'imblearn unavailable ({_e}); will fall back to noise-only augmentation.')
    HAS_SMOTE = False


def gaussian_noise_augment(X: np.ndarray, sigma: float = 0.01) -> np.ndarray:
    """Add Gaussian noise scaled by per-feature std for regularisation."""
    feat_std = X.std(axis=0, keepdims=True).clip(1e-6)
    noise    = np.random.default_rng(7).normal(0, sigma, X.shape).astype(np.float32)
    X_noisy  = X + noise * feat_std
    return np.clip(X_noisy, X.min(axis=0), X.max(axis=0))


def smote_minority(X: np.ndarray, y: np.ndarray, target_floor: int | None = None,
                   k_neighbors: int = 5, random_state: int = 42):
    """SMOTE oversample classes below target_floor (default = median class size)."""
    counts = Counter(y.tolist())
    if target_floor is None:
        target_floor = int(np.median(list(counts.values())))
    sampling_strategy = {
        cls: max(cnt, target_floor) for cls, cnt in counts.items()
        if cnt < target_floor
    }
    if not sampling_strategy:
        print('SMOTE: all classes already above target floor — skipping.')
        return X, y
    # k_neighbors must be < smallest minority count
    min_cnt = min(counts[c] for c in sampling_strategy.keys())
    k = max(1, min(k_neighbors, min_cnt - 1))
    sm = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=k, random_state=random_state)
    print(f'SMOTE: floor={target_floor:,}, k_neighbors={k}, lifting classes {sorted(sampling_strategy.keys())}')
    X_res, y_res = sm.fit_resample(X, y)
    print(f'  before={dict(sorted(counts.items()))}')
    print(f'  after ={dict(sorted(Counter(y_res.tolist()).items()))}')
    return X_res.astype(np.float32), y_res.astype(y.dtype)


def augment_minority_then_noise(X: np.ndarray, y: np.ndarray, sigma: float = 0.01,
                                target_floor: int | None = None):
    """Step 1 SMOTE minority lift; Step 2 Gaussian noise on attack samples."""
    if HAS_SMOTE:
        X_smote, y_smote = smote_minority(X, y, target_floor=target_floor)
    else:
        X_smote, y_smote = X, y

    mask = y_smote != 0  # noise only on attack rows
    X_noise_part = gaussian_noise_augment(X_smote[mask], sigma)
    X_aug = np.vstack([X_smote, X_noise_part])
    y_aug = np.hstack([y_smote, y_smote[mask]])
    print(f'Hybrid augment total: {len(X_aug):,} samples '
          f'(SMOTE: {len(X_smote)-len(X):,} synth, Noise: +{X_noise_part.shape[0]:,})')
    return X_aug, y_aug


# Lift minority classes (UDP-lag ~1.6k, SSDP/NetBIOS/SNMP ~2.6-3k) up to ~6k each.
MINORITY_FLOOR = 6000
X_train_aug, y_train_aug = augment_minority_then_noise(
    X_train, y_train, sigma=0.01, target_floor=MINORITY_FLOOR
)


Noise augment: +126,564 samples → total 379,692


In [17]:
# ── Cell 8.5: Optuna tuning for XGBoost (run-once/reuse) ─────────────────────
import optuna
from sklearn.model_selection import train_test_split

N_TRIALS_XGB = 20
TIMEOUT_XGB_SEC = 1200
TOP_K_XGB = 3

# Workflow controls (override via Cell 1.5 when present)
RUN_XGB_TUNING = globals().get('RUN_XGB_TUNING', True)
LOAD_XGB_TUNED = globals().get('LOAD_XGB_TUNED', True)
SAVE_XGB_TUNED = globals().get('SAVE_XGB_TUNED', True)
XGB_TUNED_PATH = MODELS_DIR / 'xgb_tuned_configs.json'

loaded_ok = False
if LOAD_XGB_TUNED and XGB_TUNED_PATH.exists():
    try:
        with open(XGB_TUNED_PATH, 'r') as f:
            loaded = json.load(f)
        if isinstance(loaded, dict) and 'configs' in loaded and len(loaded['configs']) > 0:
            XGB_HP_CONFIGS_OVERRIDE = loaded['configs']
            loaded_ok = True
            print(f'Loaded tuned XGB configs from {XGB_TUNED_PATH}')
            print(f"  source_time={loaded.get('saved_at','unknown')} | best_macro_f1={loaded.get('best_value','n/a')}")
            for i, hp in enumerate(XGB_HP_CONFIGS_OVERRIDE, start=1):
                print(f'  {i}. {hp}')
    except Exception as e:
        print(f'Failed to load tuned XGB configs: {e}')

if (not loaded_ok) and RUN_XGB_TUNING:
    # Use augmented train only; keep test untouched for final evaluation.
    X_tune_tr, X_tune_val, y_tune_tr, y_tune_val = train_test_split(
        X_train_aug, y_train_aug, test_size=0.2, stratify=y_train_aug, random_state=42
    )

    print(f'XGB tuning split: train={X_tune_tr.shape}, val={X_tune_val.shape}')

    def _fit_eval_xgb_trial(trial):
        hp = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
            'max_depth': trial.suggest_int('max_depth', 6, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 12.0),
            'gamma': trial.suggest_float('gamma', 0.0, 3.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 2.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }

        labels = sorted(np.unique(np.concatenate([y_tune_tr, y_tune_val])))
        lbl2idx = {l: i for i, l in enumerate(labels)}
        idx2lbl = {i: l for l, i in lbl2idx.items()}

        ytr = np.array([lbl2idx[l] for l in y_tune_tr], dtype=np.int32)
        yva = np.array([lbl2idx[l] for l in y_tune_val], dtype=np.int32)
        n_cls = len(labels)

        cls_counts = np.bincount(ytr, minlength=n_cls).astype(np.float32)
        cls_counts = np.clip(cls_counts, 1.0, None)
        cls_weights = cls_counts.sum() / (n_cls * cls_counts)
        wtr = cls_weights[ytr]

        dtr = xgb.DMatrix(X_tune_tr, label=ytr, weight=wtr)
        dva = xgb.DMatrix(X_tune_val, label=yva)

        params = {
            'device': XGB_DEVICE, 'tree_method': 'hist',
            'objective': 'multi:softprob', 'num_class': n_cls,
            'eval_metric': 'mlogloss', 'seed': 42, 'verbosity': 0,
            'max_depth': hp['max_depth'],
            'learning_rate': hp['learning_rate'],
            'subsample': hp['subsample'],
            'colsample_bytree': hp['colsample_bytree'],
            'min_child_weight': hp['min_child_weight'],
            'gamma': hp['gamma'],
            'reg_alpha': hp['reg_alpha'],
            'reg_lambda': hp['reg_lambda'],
        }

        booster = xgb.train(
            params, dtr,
            num_boost_round=hp['n_estimators'],
            evals=[(dva, 'val')],
            callbacks=[xgb.callback.EarlyStopping(rounds=30, save_best=True)],
            verbose_eval=False,
        )

        raw = booster.predict(dva)
        yproba = raw.reshape(-1, n_cls) if raw.ndim == 1 else raw
        ypred = np.argmax(yproba, axis=1)
        ypred_lbl = np.array([idx2lbl[i] for i in ypred], dtype=np.int32)

        score = f1_score(
            y_tune_val, ypred_lbl,
            labels=sorted(set(np.unique(y_tune_tr).tolist()) | set(np.unique(y_tune_val).tolist())),
            average='macro', zero_division=0
        )
        return float(score)

    study = optuna.create_study(direction='maximize', study_name='xgb_macro_f1')
    study.optimize(_fit_eval_xgb_trial, n_trials=N_TRIALS_XGB, timeout=TIMEOUT_XGB_SEC, show_progress_bar=False)

    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -1, reverse=True)[:TOP_K_XGB]
    XGB_HP_CONFIGS_OVERRIDE = []
    for t in top_trials:
        p = t.params
        XGB_HP_CONFIGS_OVERRIDE.append({
            'n_estimators': int(p['n_estimators']),
            'max_depth': int(p['max_depth']),
            'learning_rate': float(p['learning_rate']),
            'subsample': float(p['subsample']),
            'colsample_bytree': float(p['colsample_bytree']),
            'min_child_weight': float(p['min_child_weight']),
            'gamma': float(p['gamma']),
            'reg_alpha': float(p['reg_alpha']),
            'reg_lambda': float(p['reg_lambda']),
        })

    print(f'Best XGB trial macro_f1={study.best_value:.4f}')
    print('Using tuned configs (top-k):')
    for i, hp in enumerate(XGB_HP_CONFIGS_OVERRIDE, start=1):
        print(f'  {i}. {hp}')

    if SAVE_XGB_TUNED:
        payload = {
            'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            'best_value': float(study.best_value),
            'n_trials': int(len(study.trials)),
            'configs': XGB_HP_CONFIGS_OVERRIDE,
        }
        with open(XGB_TUNED_PATH, 'w') as f:
            json.dump(payload, f, indent=2)
        print(f'Saved tuned XGB configs to {XGB_TUNED_PATH}')

if 'XGB_HP_CONFIGS_OVERRIDE' not in globals():
    print('No tuned XGB override loaded/generated. Cell 9 will use default configs.')

[I 2026-05-06 16:08:39,564] A new study created in memory with name: xgb_macro_f1


XGB tuning split: train=(303753, 22), val=(75939, 22)


[I 2026-05-06 16:09:00,717] Trial 0 finished with value: 0.8882770791371363 and parameters: {'n_estimators': 700, 'max_depth': 12, 'learning_rate': 0.04640473647923862, 'subsample': 0.9502051297632758, 'colsample_bytree': 0.9524334485092751, 'min_child_weight': 8.340704443922021, 'gamma': 1.5234423066938207, 'reg_alpha': 0.0007485842269146189, 'reg_lambda': 0.006451511416162812}. Best is trial 0 with value: 0.8882770791371363.
[I 2026-05-06 16:09:16,587] Trial 1 finished with value: 0.8882055738419367 and parameters: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.032834464588970845, 'subsample': 0.8035150637353476, 'colsample_bytree': 0.738968537189103, 'min_child_weight': 7.821238782798251, 'gamma': 0.8062476129453662, 'reg_alpha': 8.403123394938472e-05, 'reg_lambda': 0.0011107308336016422}. Best is trial 0 with value: 0.8882770791371363.
[I 2026-05-06 16:09:30,325] Trial 2 finished with value: 0.8872855774234093 and parameters: {'n_estimators': 700, 'max_depth': 10, 'learn

Best XGB trial macro_f1=0.8893
Using tuned configs (top-k):
  1. {'n_estimators': 500, 'max_depth': 11, 'learning_rate': 0.08152186245240897, 'subsample': 0.8005079185246815, 'colsample_bytree': 0.8936563409503029, 'min_child_weight': 6.136443547324926, 'gamma': 0.4842468668680566, 'reg_alpha': 1.0565321870219942e-05, 'reg_lambda': 0.04009217050761578}
  2. {'n_estimators': 500, 'max_depth': 11, 'learning_rate': 0.07425219844607794, 'subsample': 0.9527336159374232, 'colsample_bytree': 0.9456613162326417, 'min_child_weight': 8.210221982285304, 'gamma': 0.5330918381546486, 'reg_alpha': 1.118514457099502e-05, 'reg_lambda': 0.03147686585112295}
  3. {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05659772927487723, 'subsample': 0.7420913378472839, 'colsample_bytree': 0.7976870217250744, 'min_child_weight': 5.800307021614034, 'gamma': 0.8084440790385001, 'reg_alpha': 0.14158571266688214, 'reg_lambda': 0.09190382811099433}
Saved tuned XGB configs to /kaggle/working/pad_onap_v3/model

In [18]:
# ── Cell 9: XGBoost training ──────────────────────────────────────────────────
TARGET_XGB = {'accuracy': 0.9887, 'macro_f1': 0.90, 'balanced_acc': 0.90}

# Deeper / regularised configs help separate confusable UDP-reflection classes.
XGB_HP_CONFIGS = [
    dict(n_estimators=500, max_depth=10, learning_rate=0.05, subsample=0.85,
         colsample_bytree=0.85, min_child_weight=1.0, gamma=0.0, reg_lambda=1.0),
    dict(n_estimators=700, max_depth=12, learning_rate=0.03, subsample=0.9,
         colsample_bytree=0.85, min_child_weight=1.0, gamma=0.0, reg_lambda=1.5),
    dict(n_estimators=900, max_depth=10, learning_rate=0.03, subsample=0.9,
         colsample_bytree=0.9,  min_child_weight=2.0, gamma=0.5, reg_lambda=2.0),
]

if 'XGB_HP_CONFIGS_OVERRIDE' in globals() and len(XGB_HP_CONFIGS_OVERRIDE) > 0:
    XGB_HP_CONFIGS = XGB_HP_CONFIGS_OVERRIDE
    print(f'Using tuned XGB configs from Optuna: {len(XGB_HP_CONFIGS)} trial(s).')

# Power scaling — emphasise hardest classes more strongly than vanilla balanced weights.
# weight_c = (N / (n_cls * count_c)) ** WEIGHT_POWER
# WEIGHT_POWER=1.0 = standard balanced; >1 = stronger focus on minority.
WEIGHT_POWER = 1.25
# Extra multiplier for known-hard classes (DrDoS_SSDP=8, UDP-lag=11, DrDoS_NetBIOS=4)
HARD_CLASS_BOOST = {4: 1.3, 8: 1.5, 11: 1.6}


def _compute_class_weights(y_tr_r: np.ndarray, n_cls: int, idx2lbl: dict) -> np.ndarray:
    cls_counts = np.bincount(y_tr_r, minlength=n_cls).astype(np.float32)
    cls_counts = np.clip(cls_counts, 1.0, None)
    base = cls_counts.sum() / (n_cls * cls_counts)
    w = np.power(base, WEIGHT_POWER)
    for ridx, orig_lbl in idx2lbl.items():
        if orig_lbl in HARD_CLASS_BOOST:
            w[ridx] *= HARD_CLASS_BOOST[orig_lbl]
    return w


def train_xgboost(X_tr, y_tr, X_te, y_te, hp, device):
    all_labels = sorted(np.unique(np.concatenate([y_tr, y_te])))
    n_cls = len(all_labels)
    lbl2idx = {l: i for i, l in enumerate(all_labels)}
    idx2lbl = {i: l for l, i in lbl2idx.items()}

    y_tr_r = np.array([lbl2idx[l] for l in y_tr], dtype=np.int32)
    y_te_r = np.array([lbl2idx[l] for l in y_te], dtype=np.int32)

    cls_weights = _compute_class_weights(y_tr_r, n_cls, idx2lbl)
    w_tr = cls_weights[y_tr_r]
    print(f'  class weights (power={WEIGHT_POWER}, hard_boost={HARD_CLASS_BOOST}):')
    for ri, ol in idx2lbl.items():
        print(f'    {CLASS_NAMES.get(ol, ol):<15} (idx {ri}) -> w={cls_weights[ri]:.3f}')

    dtrain = xgb.DMatrix(X_tr, label=y_tr_r, weight=w_tr)
    dtest = xgb.DMatrix(X_te, label=y_te_r)

    params = {
        'device': device, 'tree_method': 'hist',
        'objective': 'multi:softprob', 'num_class': n_cls,
        'eval_metric': 'mlogloss', 'seed': 42, 'verbosity': 0,
        'max_depth':        hp.get('max_depth',        10),
        'learning_rate':    hp.get('learning_rate',    0.05),
        'subsample':        hp.get('subsample',        0.85),
        'colsample_bytree': hp.get('colsample_bytree', 0.85),
        'min_child_weight': hp.get('min_child_weight', 1.0),
        'gamma':            hp.get('gamma',            0.0),
        'reg_alpha':        hp.get('reg_alpha',        0.0),
        'reg_lambda':       hp.get('reg_lambda',       1.0),
    }
    t0 = time.time()
    booster = xgb.train(
        params, dtrain,
        num_boost_round=hp.get('n_estimators', 500),
        evals=[(dtest, 'eval')],
        callbacks=[xgb.callback.EarlyStopping(rounds=40, save_best=True)],
        verbose_eval=100,
    )
    elapsed = time.time() - t0

    raw = booster.predict(dtest)
    y_proba_r = raw.reshape(-1, n_cls) if raw.ndim == 1 else raw
    y_pred_r = np.argmax(y_proba_r, axis=1)
    y_pred = np.array([idx2lbl[i] for i in y_pred_r], dtype=np.int32)

    y_proba = np.zeros((len(y_te), N_CLASSES), dtype=np.float32)
    for ri, ol in idx2lbl.items():
        if ol < N_CLASSES:
            y_proba[:, ol] = y_proba_r[:, ri]

    acc = float(accuracy_score(y_te, y_pred))
    bal_acc = float(balanced_accuracy_score(y_te, y_pred))
    all_classes = sorted(set(np.unique(y_tr).tolist()) | set(np.unique(y_te).tolist()))
    macro_f1 = float(f1_score(y_te, y_pred, labels=all_classes, average='macro', zero_division=0))

    classes_present = np.unique(y_te).tolist()
    try:
        y_bin = label_binarize(y_te, classes=list(range(N_CLASSES)))
        auc = float(roc_auc_score(
            y_bin[:, classes_present], y_proba[:, classes_present],
            multi_class='ovr', average='macro'
        ))
    except Exception:
        auc = 0.0

    booster._label_to_idx = lbl2idx
    booster._idx_to_label = idx2lbl
    booster._n_classes = n_cls
    metrics = {
        'accuracy': acc,
        'balanced_acc': bal_acc,
        'macro_f1': macro_f1,
        'auc_macro_ovr': auc,
    }
    return booster, metrics, elapsed, y_pred, y_proba


best_xgb, best_xgb_metrics, best_y_pred, best_y_proba = None, {}, None, None

for attempt, hp in enumerate(XGB_HP_CONFIGS):
    print(f'\n--- Attempt {attempt+1}/{len(XGB_HP_CONFIGS)}: {hp} ---')
    booster, metrics, elapsed, y_pred, y_proba = train_xgboost(
        X_train_aug, y_train_aug, X_test, y_test, hp, XGB_DEVICE
    )
    print(
        f'  accuracy={metrics["accuracy"]:.4f}  balanced_acc={metrics["balanced_acc"]:.4f}  '
        f'macro_f1={metrics["macro_f1"]:.4f}  auc={metrics["auc_macro_ovr"]:.4f}  time={elapsed:.1f}s',
    )

    if not best_xgb_metrics or metrics['macro_f1'] > best_xgb_metrics.get('macro_f1', 0):
        best_xgb, best_xgb_metrics, best_y_pred, best_y_proba = booster, metrics, y_pred, y_proba

    if all(metrics.get(k, 0) >= v for k, v in TARGET_XGB.items()):
        print('All targets met!')
        break
    else:
        missing = [k for k, v in TARGET_XGB.items() if metrics.get(k, 0) < v]
        print(f'  Missing: {missing} - trying next config...')

print(
    f'\nBEST XGBoost: accuracy={best_xgb_metrics["accuracy"]:.4f}  '
    f'balanced_acc={best_xgb_metrics["balanced_acc"]:.4f}  '
    f'macro_f1={best_xgb_metrics["macro_f1"]:.4f}  auc={best_xgb_metrics["auc_macro_ovr"]:.4f}'
)


Using tuned XGB configs from Optuna: 3 trial(s).

--- Attempt 1/3: {'n_estimators': 500, 'max_depth': 11, 'learning_rate': 0.08152186245240897, 'subsample': 0.8005079185246815, 'colsample_bytree': 0.8936563409503029, 'min_child_weight': 6.136443547324926, 'gamma': 0.4842468668680566, 'reg_alpha': 1.0565321870219942e-05, 'reg_lambda': 0.04009217050761578} ---
[0]	eval-mlogloss:2.05891
[100]	eval-mlogloss:0.35619
[125]	eval-mlogloss:0.35822
  accuracy=0.8457  balanced_acc=0.7969  macro_f1=0.7817  auc=0.9795  time=8.7s
  Missing: ['accuracy', 'macro_f1', 'balanced_acc'] - trying next config...

--- Attempt 2/3: {'n_estimators': 500, 'max_depth': 11, 'learning_rate': 0.07425219844607794, 'subsample': 0.9527336159374232, 'colsample_bytree': 0.9456613162326417, 'min_child_weight': 8.210221982285304, 'gamma': 0.5330918381546486, 'reg_alpha': 1.118514457099502e-05, 'reg_lambda': 0.03147686585112295} ---
[0]	eval-mlogloss:2.08496
[100]	eval-mlogloss:0.35445
[146]	eval-mlogloss:0.35474
  accurac

In [19]:
# ── Cell 10: XGBoost full eval + confusion matrix + SHAP ─────────────────────
present_classes = sorted(np.unique(np.concatenate([y_train, y_test])))
target_names = [CLASS_NAMES[i] for i in present_classes]

print('=== CLASSIFICATION REPORT ===')
print(classification_report(
    y_test, best_y_pred,
    labels=present_classes, target_names=target_names,
    digits=4, zero_division=0
))

# PR-AUC (one-vs-rest) gives better signal on imbalanced setups
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
pr_auc_per_class = {}
for cls in present_classes:
    try:
        pr_auc_per_class[cls] = float(average_precision_score(y_test_bin[:, cls], best_y_proba[:, cls]))
    except Exception:
        pr_auc_per_class[cls] = 0.0

print('\n=== PR-AUC (OvR) ===')
for cls in present_classes:
    print(f'  {CLASS_NAMES[cls]:<15}: {pr_auc_per_class[cls]:.4f}')
print(f"  Macro PR-AUC     : {np.mean([pr_auc_per_class[c] for c in present_classes]):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, best_y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(present_classes)), yticks=range(len(present_classes)),
    xticklabels=target_names, yticklabels=target_names,
    title='XGBoost Confusion Matrix', ylabel='True', xlabel='Predicted'
    )
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
thresh = cm.max() / 2
for i in range(len(present_classes)):
    for j in range(len(present_classes)):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_xgb_confusion_matrix.png', dpi=150)
plt.show()

# SHAP
print('\nComputing SHAP values (sample=500)...')
try:
    explainer = shap.TreeExplainer(best_xgb)
    X_shap = X_test[:500]
    shap_out = explainer(X_shap)
    sv = shap_out.values
    if sv.ndim == 3:
        mean_shap = np.abs(sv).mean(axis=(0, 2))
        shap_values = [sv[:, :, c] for c in range(sv.shape[2])]
    else:
        mean_shap = np.abs(sv).mean(axis=0)
        shap_values = [sv]

    top5_idx = np.argsort(mean_shap)[-5:][::-1]
    print('Top-5 SHAP features:')
    for i in top5_idx:
        print(f'  {FEATURE_NAMES[i]}: {mean_shap[i]:.4f}')

    plt.figure(figsize=(10, 6))
    shap_class_names = [CLASS_NAMES.get(best_xgb._idx_to_label[i], f'class_{i}')
                        for i in range(best_xgb._n_classes)]
    shap.summary_plot(
        shap_values, X_shap, feature_names=FEATURE_NAMES,
        class_names=shap_class_names, show=False, max_display=22
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_shap_summary.png', dpi=150)
    plt.show()
except Exception as e:
    print(f'SHAP error: {e}')

# Save XGBoost model
xgb_path = MODELS_DIR / 'xgboost_v3.json'
best_xgb.save_model(str(xgb_path))
with open(MODELS_DIR / 'xgb_label_map.json', 'w') as f:
    json.dump({
        'label_to_idx': {str(int(k)): int(v) for k, v in best_xgb._label_to_idx.items()},
        'idx_to_label': {str(int(k)): int(v) for k, v in best_xgb._idx_to_label.items()},
        'n_classes': int(best_xgb._n_classes),
        'metrics': {
            k: (float(v) if hasattr(v, '__float__') else
                [float(x) for x in v] if isinstance(v, list) else v)
            for k, v in best_xgb_metrics.items()
        }
    }, f, indent=2)
print(f'\nSaved: {xgb_path}')

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

       BENIGN     0.9988    0.9993    0.9990      4148
    DrDoS_DNS     0.9843    0.9921    0.9882      2657
   DrDoS_LDAP     0.9932    0.9861    0.9896      4895
  DrDoS_MSSQL     1.0000    0.9967    0.9984      3998
DrDoS_NetBIOS     0.9953    0.8711    0.9291      3181
    DrDoS_NTP     1.0000    1.0000    1.0000      2657
   DrDoS_SNMP     0.8525    0.9616    0.9038      2657
   DrDoS_SSDP     0.3940    0.2680    0.3190      2657
    DrDoS_UDP     0.6098    0.7412    0.6691      4046
          Syn     0.6147    0.9025    0.7313      2657
      UDP-lag     0.2174    0.0563    0.0895      1598

     accuracy                         0.8471     35151
    macro avg     0.7873    0.7977    0.7834     35151
 weighted avg     0.8307    0.8471    0.8324     35151


=== PR-AUC (OvR) ===
  BENIGN         : 1.0000
  DrDoS_DNS      : 0.9993
  DrDoS_LDAP     : 0.9992
  DrDoS_MSSQL    : 1.0000
  DrDoS_NetBIOS 

---
## Section 4 — Track B: LSTM 3-Horizon Forecaster

In [15]:
# ── Cell 7.5: Track B time-series aggregation (1-min) ─────────────────────────
TS_FEATURE_NAMES = [
    'pkt_count_total',
    'byte_count_total',
    'unique_src_ip_count',
    'unique_dst_ip_count',
    'avg_pkt_size',
    'syn_count',
]
LOOKBACK_MINUTES = 60
HORIZONS_MINUTES = [1, 5, 15]
MAX_TS_MINUTES = None  # set to int to cap the number of minutes
TS_CHUNKSIZE = 300_000


def _num_col(df: pd.DataFrame, name: str, default: float = 0.0) -> pd.Series:
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(default)
    return pd.Series(np.full(len(df), default))


def _aggregate_minute_block(df: pd.DataFrame, X_rows, y_rows, ts_rows):
    if len(df) == 0:
        return X_rows, y_rows, ts_rows

    df = df.copy()
    fwd_pkts  = _num_col(df, 'Total Fwd Packets', 0.0)
    bwd_pkts  = _num_col(df, 'Total Backward Packets', 0.0)
    fwd_bytes = _num_col(df, 'Total Length of Fwd Packets', 0.0)
    bwd_bytes = _num_col(df, 'Total Length of Bwd Packets', 0.0)
    syn_cnt   = _num_col(df, 'SYN Flag Count', 0.0)

    df['total_pkts']  = fwd_pkts + bwd_pkts
    df['total_bytes'] = fwd_bytes + bwd_bytes
    df['syn_count']   = syn_cnt
    has_src = 'Source IP' in df.columns
    has_dst = 'Destination IP' in df.columns

    for minute, g in df.groupby('minute'):
        pkt_count_total  = float(g['total_pkts'].sum())
        byte_count_total = float(g['total_bytes'].sum())
        if has_src:
            unique_src_ip_count = int(g['Source IP'].nunique())
        else:
            unique_src_ip_count = 0
        if has_dst:
            unique_dst_ip_count = int(g['Destination IP'].nunique())
        else:
            unique_dst_ip_count = 0
        avg_pkt_size = byte_count_total / max(pkt_count_total, 1.0)
        syn_count = float(g['syn_count'].sum())
        attack_flag = int((g['_class'] != 0).any())

        X_rows.append([
            pkt_count_total,
            byte_count_total,
            unique_src_ip_count,
            unique_dst_ip_count,
            avg_pkt_size,
            syn_count,
        ])
        y_rows.append(attack_flag)
        ts_rows.append(pd.Timestamp(minute).value)
        if MAX_TS_MINUTES and len(X_rows) >= MAX_TS_MINUTES:
            break
    return X_rows, y_rows, ts_rows


def build_track_b_timeseries(csv_files):
    X_rows, y_rows, ts_rows = [], [], []
    for f in csv_files:
        carry_over = pd.DataFrame()
        try:
            for chunk in pd.read_csv(f, chunksize=TS_CHUNKSIZE, low_memory=False, on_bad_lines='skip'):
                chunk.columns = [c.strip() for c in chunk.columns]
                lcol = next((c for c in chunk.columns if c.lower() == 'label'), None)
                tcol = next((c for c in chunk.columns if c.lower() == 'timestamp'), None)
                if lcol is None or tcol is None:
                    continue

                chunk[tcol] = pd.to_datetime(chunk[tcol], errors='coerce')
                chunk = chunk[chunk[tcol].notna()]
                if len(chunk) == 0:
                    continue

                mapped = chunk[lcol].str.strip().map(LABEL_MAP)
                chunk['_class'] = mapped
                chunk = chunk.dropna(subset=['_class'])
                if len(chunk) == 0:
                    continue
                chunk['_class'] = chunk['_class'].astype(int)
                chunk['minute'] = chunk[tcol].dt.floor('min')

                chunk = pd.concat([carry_over, chunk], ignore_index=True)
                if len(chunk) == 0:
                    continue

                last_minute = chunk['minute'].iloc[-1]
                complete = chunk[chunk['minute'] < last_minute]
                carry_over = chunk[chunk['minute'] == last_minute]

                X_rows, y_rows, ts_rows = _aggregate_minute_block(
                    complete, X_rows, y_rows, ts_rows)
                if MAX_TS_MINUTES and len(X_rows) >= MAX_TS_MINUTES:
                    break

            if len(carry_over) > 0 and (not MAX_TS_MINUTES or len(X_rows) < MAX_TS_MINUTES):
                X_rows, y_rows, ts_rows = _aggregate_minute_block(
                    carry_over, X_rows, y_rows, ts_rows)
        except Exception as e:
            print(f'  ERROR {os.path.basename(f)}: {e}')

        if MAX_TS_MINUTES and len(X_rows) >= MAX_TS_MINUTES:
            break

    if len(X_rows) == 0:
        raise ValueError('No Track B time-series data extracted.')

    order = np.argsort(ts_rows)
    X_ts = np.array([X_rows[i] for i in order], dtype=np.float32)
    y_ts = np.array([y_rows[i] for i in order], dtype=np.int32)
    return X_ts, y_ts


def build_sequences(X_ts, y_ts, lookback, horizons):
    X_seq, y_seq = [], []
    max_h = max(horizons)
    for i in range(lookback, len(X_ts) - max_h):
        X_seq.append(X_ts[i - lookback:i])
        y_seq.append([y_ts[i + h] for h in horizons])
    if len(X_seq) == 0:
        raise ValueError('Not enough time-series points for Track B sequences.')
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


print('Building Track B time-series (1-min) ...')
X_ts_raw, y_ts_raw = build_track_b_timeseries(csv_files)
print(f'Track B minutes: X={X_ts_raw.shape} y={y_ts_raw.shape}')

X_ts_seq, y_ts_seq = build_sequences(X_ts_raw, y_ts_raw, LOOKBACK_MINUTES, HORIZONS_MINUTES)
print(f'Track B sequences: X={X_ts_seq.shape} y={y_ts_seq.shape}')

# Chronological split 70/15/15
n_total = len(X_ts_seq)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

X_ts_train_seq = X_ts_seq[:n_train]
y_ts_train_seq = y_ts_seq[:n_train]
X_ts_val_seq   = X_ts_seq[n_train:n_train + n_val]
y_ts_val_seq   = y_ts_seq[n_train:n_train + n_val]
X_ts_test_seq  = X_ts_seq[n_train + n_val:]
y_ts_test_seq  = y_ts_seq[n_train + n_val:]

# Min-Max scaling per feature (fit on train only)
ts_scaler = MinMaxScaler()
train_flat = X_ts_train_seq.reshape(-1, X_ts_train_seq.shape[-1])
ts_scaler.fit(train_flat)

def _scale_seq(X):
    flat = X.reshape(-1, X.shape[-1])
    flat = ts_scaler.transform(flat)
    return flat.reshape(X.shape)

X_ts_train_seq = _scale_seq(X_ts_train_seq)
X_ts_val_seq   = _scale_seq(X_ts_val_seq)
X_ts_test_seq  = _scale_seq(X_ts_test_seq)

print('Track B split:')
print(f'  Train: {X_ts_train_seq.shape}  Val: {X_ts_val_seq.shape}  Test: {X_ts_test_seq.shape}')
print(f'  Attack rate (train): {y_ts_train_seq.mean(axis=0)}')

Building Track B time-series (1-min) ...


KeyboardInterrupt: 

In [ ]:
# ── Cell 11: Model definition — LSTM (3 horizons) ───────────────────────────
N_TIMESTEPS    = LOOKBACK_MINUTES
N_FEATURES     = len(TS_FEATURE_NAMES)
N_HORIZONS     = len(HORIZONS_MINUTES)
HORIZON_LABELS = [f'+{h}m' for h in HORIZONS_MINUTES]

if 'X_ts_train_seq' not in globals():
    raise ValueError('Track B time-series not found. Run Cell 7.5 first.')


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMForecaster(nn.Module):
    """Stacked LSTM forecaster with 3 sigmoid heads (t+1/t+5/t+15)."""
    def __init__(self, input_dim=N_FEATURES, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, 100, batch_first=True)
        self.lstm2 = nn.LSTM(100, 100, batch_first=True)
        self.lstm3 = nn.LSTM(100, 50, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(50, 25)
        self.fc2 = nn.Linear(25, N_HORIZONS)

    def forward(self, x):
        h, _ = self.lstm1(x)
        h = self.dropout(h)
        h, _ = self.lstm2(h)
        h = self.dropout(h)
        h, _ = self.lstm3(h)
        h = self.dropout(h[:, -1, :])
        h = torch.relu(self.fc1(h))
        return self.fc2(h)


print(f'Model defined. Device: {DEVICE}')
_m = LSTMForecaster().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m


In [ ]:
# ── Cell 11.5: LSTM training config ─────────────────────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 100
PATIENCE     = 10
LR           = 1e-3
POS_WEIGHT   = 5.0
LOSS_WEIGHTS = [0.5, 1.0, 1.5]
TARGET_LSTM  = {'auc_h0': 0.80}


In [ ]:
# ── Cell 12: LSTM training (binary multi-horizon) ───────────────────────────
train_ds = SequenceDataset(X_ts_train_seq, y_ts_train_seq)
val_ds   = SequenceDataset(X_ts_val_seq,   y_ts_val_seq)
test_ds  = SequenceDataset(X_ts_test_seq,  y_ts_test_seq)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

model = LSTMForecaster().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
loss_weights = torch.tensor(LOSS_WEIGHTS, dtype=torch.float32, device=DEVICE)
pos_weight = torch.tensor([POS_WEIGHT], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')

def compute_loss(logits, labels):
    loss = criterion(logits, labels)
    loss = loss * loss_weights
    return loss.mean()

best_state = None
best_val = float('inf')
patience_ct = 0
train_losses, val_losses = [], []
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for Xb, yb in train_dl:
        Xb = Xb.to(DEVICE)
        yb = yb.to(DEVICE)
        logits = model(Xb)
        loss = compute_loss(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        total += loss.item()
    train_losses.append(total / max(len(train_dl), 1))

    model.eval()
    with torch.no_grad():
        vtotal = 0.0
        for Xb, yb in val_dl:
            Xb = Xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(Xb)
            vtotal += compute_loss(logits, yb).item()
        val_loss = vtotal / max(len(val_dl), 1)
        val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f}')

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_ct = 0
    else:
        patience_ct += 1
        if patience_ct >= PATIENCE:
            print(f'Early stop at epoch {epoch+1}')
            break

if best_state:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# Test evaluation
model.eval()
probs_all, labels_all = [], []
with torch.no_grad():
    for Xb, yb in test_dl:
        Xb = Xb.to(DEVICE)
        logits = model(Xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_all.append(probs)
        labels_all.append(yb.numpy())

probs_all = np.vstack(probs_all)
labels_all = np.vstack(labels_all)

auc_roc, auprc = [], []
for h in range(N_HORIZONS):
    y_true = labels_all[:, h]
    y_prob = probs_all[:, h]
    try:
        auc_roc.append(float(roc_auc_score(y_true, y_prob)))
    except Exception:
        auc_roc.append(0.0)
    try:
        auprc.append(float(average_precision_score(y_true, y_prob)))
    except Exception:
        auprc.append(0.0)

best_lstm = model
best_lstm_metrics = {
    'auc_roc': auc_roc,
    'auprc': auprc,
    'val_loss': float(best_val),
    'train_loss': float(train_losses[-1]) if train_losses else 0.0,
}
best_lstm_train_losses = train_losses
best_lstm_val_losses = val_losses

elapsed = time.time() - t0
print('LSTM evaluation:')
for h in range(N_HORIZONS):
    print(f'  {HORIZON_LABELS[h]}: AUC={auc_roc[h]:.4f}  AUPRC={auprc[h]:.4f}')
print(f'Time: {elapsed:.1f}s')


In [ ]:
# ── Cell 13: LSTM training curves + save ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(best_lstm_train_losses, label='train', color='steelblue')
axes[0].plot(best_lstm_val_losses, label='val', color='coral')
axes[0].set_title('Train vs Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

auc_vals = best_lstm_metrics['auc_roc']
bars = axes[1].bar(HORIZON_LABELS, auc_vals, color=['#2196F3','#4CAF50','#FF9800'])
axes[1].axhline(TARGET_LSTM['auc_h0'], color='red', linestyle='--', label=f'Target {TARGET_LSTM["auc_h0"]}')
axes[1].set_ylim(0.5, 1.0); axes[1].set_title('AUC-ROC per Horizon')
axes[1].set_ylabel('AUC'); axes[1].legend()
for bar, v in zip(bars, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.4f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_lstm_training.png', dpi=150)
plt.show()

# Save LSTM model
lstm_path = MODELS_DIR / 'lstm_v3.pt'
torch.save(best_lstm.state_dict(), str(lstm_path))
with open(MODELS_DIR / 'lstm_metrics.json', 'w') as f:
    json.dump(best_lstm_metrics, f, indent=2)
with open(MODELS_DIR / 'ts_scaler.pkl', 'wb') as f:
    pickle.dump(ts_scaler, f)
print(f'Saved: {lstm_path}')


---
## Section 5 — Final Summary

In [ ]:
# ── Cell 14: Final summary ───────────────────────────────────────────────────
print('=' * 65)
print('PAD-ONAP M2 - TRAINING COMPLETE')
print('=' * 65)
print()
print(f'DATASET (full scan):')
print(f'  Total rows read : {total_rows_read:>12,}')
print(f'  BENIGN rows     : {total_benign_rows:>12,}')
print(f'  Attack rows     : {grand_attack:>12,}')
print(f'  Raw BENIGN wins : {n_raw:>12,}')
print(f'  Total windows   : {len(X_train) + len(X_test):>12,}  (train+test)')
print()
print('VALIDATION SETTINGS:')
print(f'  Temporal split ratio : {TEMPORAL_SPLIT_RATIO}')
print(f'  Purge windows        : {PURGE_WINDOWS}')
print(f'  Max BENIGN oversample: x{MAX_BENIGN_OVERSAMPLE} (train only)')
print('  Test BENIGN          : no oversampling')
print()
print(f'TRACK A - XGBoost 12-class:')
print(f'  Accuracy     : {best_xgb_metrics["accuracy"]:.4f}  (target >= {TARGET_XGB["accuracy"]})')
print(f'  Balanced Acc : {best_xgb_metrics["balanced_acc"]:.4f}  (target >= {TARGET_XGB["balanced_acc"]})')
print(f'  Macro F1     : {best_xgb_metrics["macro_f1"]:.4f}  (target >= {TARGET_XGB["macro_f1"]})')
print(f'  AUC (OvR)    : {best_xgb_metrics["auc_macro_ovr"]:.4f}')
print(f'  Saved        : {MODELS_DIR}/xgboost_v3.json')
print()
print('TRACK B - LSTM 3-horizon:')
for h, (auc, pr) in enumerate(zip(best_lstm_metrics['auc_roc'], best_lstm_metrics['auprc'])):
    print(f'  {HORIZON_LABELS[h]}: AUC={auc:.4f}  AUPRC={pr:.4f}')
print(f'  Saved        : {MODELS_DIR}/lstm_v3.pt')
print()
print(f'OUTPUTS saved to: {OUTPUT_DIR}')
print()

# Check targets
a_ok = all(best_xgb_metrics.get(k, 0) >= v for k, v in TARGET_XGB.items())
b_ok = best_lstm_metrics.get('auc_roc', [0])[0] >= TARGET_LSTM['auc_h0']
print(f'Track A targets met: {"YES" if a_ok else "NO"}')
print(f'Track B targets met: {"YES" if b_ok else "NO"}')

In [ ]:
# ── Cell 15: Package outputs for download ─────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/pad_onap_v3_models', 'zip', str(MODELS_DIR))
print('Created: /kaggle/working/pad_onap_v3_models.zip')

# List all saved files
print('\nAll output files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {str(p.relative_to(OUTPUT_DIR)):<45} {size_kb:>8.1f} KB')

In [ ]:
!zip -r folder.zip /kaggle/working/pad_onap_v3